# Clinical Research: Predicting Blood Pressure Control

**Question.** Among adults with hypertension, do age, gender, palliative care and dialysis predict whether their
most recent blood pressure in 2026 is controlled (below 140/90 mmHg)?

The notebook uses three languages, each for what it does best:

1. **CQL** defines the cohort, the predictors and the outcome once, in a standard clinical language over FHIR data.
2. **SQL** shapes the CQL results into an analysis table and summarizes it.
3. **Python** fits a model, tests it on held-out patients and puts confidence intervals on each predictor.

Data: the synthetic small population (4,000 synthetic patients). No real patient data.

**Runs on serverless.** The client is already in this notebook's environment, so there is nothing to install.
Just **Run All**.

In [ ]:
# Initialize the CQL engine: the client wheel comes from this notebook's environment.
# Every %%cql cell runs in the default environment: the deploy's example patients and measures.
get_ipython().run_line_magic("load_ext", "cql_databricks.compiler")
print("CQL engine ready: %%cql cells run in the default environment")

## 1. CQL: cohort, predictors and outcome

All logic is in this one cell. Each `define` is a standard definition. CQL is an HL7 standard over FHIR, so the
same definitions can be shared with any other organization and run on their technology stack, not only this one.

| Define | Meaning |
|---|---|
| `In Study` | Has essential hypertension (ICD-10-CM I10) and is 18 to 85 at the end of 2026 |
| `Age`, `Gender` | Demographic predictors, read from the FHIR Patient resource |
| `Palliative Care` | Health predictor: a palliative care encounter diagnosis (ICD-10-CM Z51.5) |
| `Dialysis` | Health predictor: renal dialysis dependence (ICD-10-CM Z99.2) or a completed dialysis procedure (ICD-10-PCS 5A1D70Z) |
| `Most Recent Blood Pressure` | The last final blood pressure reading in 2026 |
| `BP Controlled` | The outcome: that reading is below 140/90 mmHg. No reading counts as not controlled |
| `StudyRecord` | One row per patient in the study: the predictors and the outcome together |

The result has one row per patient in the environment; patients outside the study have a null record. It lands in a
session view named after the define, so the SQL below reads `StudyRecord` directly.

In [ ]:
%%cql --target StudyRecord
library HypertensionControlStudy version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers

codesystem "ICD10CM": 'http://hl7.org/fhir/sid/icd-10-cm'
codesystem "ICD10PCS": 'http://www.cms.gov/Medicare/Coding/ICD10'
codesystem "LOINC": 'http://loinc.org'

code "Essential hypertension": 'I10' from "ICD10CM"
code "Palliative care encounter": 'Z51.5' from "ICD10CM"
code "Dependence on renal dialysis": 'Z99.2' from "ICD10CM"
code "Hemodialysis procedure": '5A1D70Z' from "ICD10PCS"
code "Blood pressure panel": '85354-9' from "LOINC"
code "Systolic blood pressure": '8480-6' from "LOINC"
code "Diastolic blood pressure": '8462-4' from "LOINC"

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Age":
  AgeInYearsAt(date from end of "Measurement Period")

define "Gender":
  Patient.gender.value

define "Palliative Care":
  exists [Condition: "Palliative care encounter"]

define "Dialysis":
  exists [Condition: "Dependence on renal dialysis"]
    or exists ([Procedure: "Hemodialysis procedure"] P where P.status = 'completed')

define "In Study":
  exists [Condition: "Essential hypertension"]
    and "Age" in Interval[18, 85]

define fluent function readingTime(reading Observation):
  if reading.effective is FHIR.Period then
    end of FHIRHelpers.ToInterval(reading.effective as FHIR.Period)
  else
    FHIRHelpers.ToDateTime(reading.effective as FHIR.dateTime)

define "Blood Pressure Readings":
  [Observation: "Blood pressure panel"] Reading
    where Reading.status in { 'final', 'amended', 'corrected' }
      and Reading.readingTime() during day of "Measurement Period"

define "Most Recent Blood Pressure":
  First("Blood Pressure Readings" Reading
    where not exists ("Blood Pressure Readings" Later
      where Later.readingTime() after Reading.readingTime())
    sort by id)

define "Most Recent Systolic":
  First("Blood Pressure Readings" Reading
    where Reading.id = "Most Recent Blood Pressure".id
    return First(Reading.component Systolic
      where Systolic.code ~ "Systolic blood pressure"
      return Systolic.value as Quantity))

define "Most Recent Diastolic":
  First("Blood Pressure Readings" Reading
    where Reading.id = "Most Recent Blood Pressure".id
    return First(Reading.component Diastolic
      where Diastolic.code ~ "Diastolic blood pressure"
      return Diastolic.value as Quantity))

define "BP Controlled":
  Coalesce("Most Recent Systolic" < 140 'mm[Hg]' and "Most Recent Diastolic" < 90 'mm[Hg]', false)

define "StudyRecord":
  if "In Study" then
    Tuple {
      age: "Age",
      gender: "Gender",
      palliative_care: "Palliative Care",
      dialysis: "Dialysis",
      bp_controlled: "BP Controlled"
    }
  else null

## 2. SQL: build the analysis table

Plain SQL over the CQL result: keep patients in the study, unpack the predictors into columns, add an age band,
and summarize the control rate by group.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW bp_study AS
SELECT __subject_id                              AS patient_id,
       __val.age                                 AS age,
       CASE WHEN __val.age < 40 THEN '18-39'
            WHEN __val.age < 65 THEN '40-64'
            ELSE '65-85' END                     AS age_band,
       __val.gender                              AS gender,
       CAST(__val.palliative_care AS INT)        AS palliative_care,
       CAST(__val.dialysis AS INT)               AS dialysis,
       CAST(__val.bp_controlled AS INT)          AS bp_controlled
  FROM StudyRecord
 WHERE __val IS NOT NULL;

SELECT 'Age ' || age_band                AS `group`, count(*) AS patients, round(100 * avg(bp_controlled), 1) AS pct_controlled
  FROM bp_study GROUP BY age_band
UNION ALL
SELECT 'Gender ' || gender, count(*), round(100 * avg(bp_controlled), 1) FROM bp_study GROUP BY gender
UNION ALL
SELECT 'Palliative care ' || IF(palliative_care = 1, 'yes', 'no'), count(*), round(100 * avg(bp_controlled), 1)
  FROM bp_study GROUP BY palliative_care
UNION ALL
SELECT 'Dialysis ' || IF(dialysis = 1, 'yes', 'no'), count(*), round(100 * avg(bp_controlled), 1)
  FROM bp_study GROUP BY dialysis
ORDER BY `group`

## 3. Python: predict control and test each predictor

Python does what SQL cannot: fit a logistic regression, score it on patients it never saw, and bootstrap a
95% confidence interval for each predictor's odds ratio.

- **AUC** is how well the model ranks controlled patients above uncontrolled ones: 0.5 is chance, 1.0 is perfect.
- **Odds ratio** above 1 means higher odds of control. An interval that crosses 1 means no clear effect.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.model_selection import train_test_split

study = spark.table("bp_study").toPandas()
X = pd.DataFrame({"age_per_10y": study["age"] / 10,
                  "male": (study["gender"] == "male").astype(float),
                  "palliative_care": study["palliative_care"].astype(float),
                  "dialysis": study["dialysis"].astype(float)})
y = study["bp_controlled"]
print(f"{len(study)} patients, {y.mean():.0%} controlled")

# Held-out test: train on 70% of patients, score the other 30%.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=7)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

# 95% confidence intervals for each odds ratio: refit on 500 bootstrap resamples of all patients.
rng = np.random.default_rng(7)
boot = []
for _ in range(500):
    i = rng.integers(0, len(X), len(X))
    boot.append(LogisticRegression(max_iter=1000).fit(X.iloc[i], y.iloc[i]).coef_[0])
full = LogisticRegression(max_iter=1000).fit(X, y).coef_[0]
odds = pd.DataFrame({"odds_ratio": np.exp(full),
                     "ci_low": np.exp(np.percentile(boot, 2.5, axis=0)),
                     "ci_high": np.exp(np.percentile(boot, 97.5, axis=0))}, index=X.columns)
odds["clear_effect"] = (odds.ci_low > 1) | (odds.ci_high < 1)

BLUE, GREY = "#2a78d6", "#8a8984"
fig, (roc_ax, or_ax) = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=roc_ax, color=BLUE, lw=2, name="Model")
roc_ax.plot([0, 1], [0, 1], ls="--", color=GREY, lw=1, label="Chance")
roc_ax.set_title(f"Held-out ROC curve (AUC {auc:.2f})")
roc_ax.legend(loc="lower right", frameon=False)

LABELS = {"age_per_10y": "Age (per 10 years)", "male": "Male", "palliative_care": "Palliative care",
          "dialysis": "Dialysis"}
ordered = odds.sort_values("odds_ratio")
ypos = np.arange(len(ordered))
or_ax.hlines(ypos, ordered.ci_low, ordered.ci_high, color=BLUE, lw=2)
or_ax.plot(ordered.odds_ratio, ypos, "o", color=BLUE, ms=8)
or_ax.axvline(1, ls="--", color=GREY, lw=1)
or_ax.set_xscale("log")
or_ax.set_xticks([0.1, 0.25, 0.5, 1, 2], ["0.1", "0.25", "0.5", "1", "2"])
or_ax.minorticks_off()
or_ax.set_xlabel("Odds ratio (log scale; below 1 = less likely controlled)")
or_ax.set_yticks(ypos, [LABELS[p] for p in ordered.index])
or_ax.set_title("Odds ratio of BP control, with 95% CI")
for ax in (roc_ax, or_ax):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

display(odds.round(2).reset_index(names="predictor"))

## What the result says

- **Age and gender:** expect intervals that cross 1. The synthesizer sets demographics independently of blood
  pressure, so this synthetic data has no link to find.
- **Palliative care and dialysis:** expect odds ratios far below 1. Read this with care: in this synthetic data,
  these patients have **no blood pressure reading at all**, and the outcome counts a missing reading as not
  controlled. The effect is missing data, not physiology. A real study would decide how to treat patients with
  no reading, and the quality measure itself excludes these patients from its denominator.
- **AUC near 0.5:** only about 20 of the 590 patients have palliative care or dialysis, so those two predictors
  barely change how the model ranks everyone else. A strong effect in a small group does not make a good
  predictor for the whole cohort.

On real data the same three cells apply unchanged: the CQL definitions stay the standard, the SQL shapes the
table, and the Python tests the hypothesis.

### Sharing the model is not enough

A model is only as portable as its inputs. Hand the fitted model to another organization and it still needs
`Dialysis`, `Palliative Care` and `BP Controlled` computed the same way. If they count dialysis from procedures
only, or take the lowest reading of the day instead of the last, the model gets different inputs. It will score
their patients without any error, and the scores will be wrong.

So share the CQL with the model. The receiving organization runs the same definitions over its own FHIR data, on
its own stack, and gets predictors that mean exactly what they meant here.

In [ ]:
import json

dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "06_clinical_research"}))